# Amparo -- M2: Evaluacion del modelo fine-tuneado

Este notebook corre el harness de evaluacion de M2 sobre el adaptador LoRA que
entreno M1 (`m1_finetune.ipynb`). Vuelve a generar las respuestas baseline y
fine-tuned sobre el **mismo split de validacion** de M1 (231 ejemplos, seed 42) y
sobre el **eval set propio** (`data/eval_set.json`, cada caso con su criterio).

Fases:
1. Generar respuestas del modelo base (baseline) y del fine-tuned. Es lo unico que
   usa la GPU.
2. Metricas sin juez: clasicas, citas numeradas y la **guardia de rutas y
   entidades** (entidades inventadas, entidades reales usadas donde no
   corresponden, conceptos fuera del tema de la pregunta).
3. Guardar todo en Drive.
4. Juez **Groq** (`openai/gpt-oss-120b`, otra familia que Qwen) sobre TODO: eval set
   contra su criterio, rubrica 1-5 de las 231 respuestas de cada modelo (con la
   lista de errores juridicos) y comparacion cara a cara de los 231 pares en los
   dos ordenes. Son ~1 000 llamadas: con el cupo gratuito (200 000 tokens al dia)
   toma ~5-7 dias con una sola key. Se retoma cada dia sin GPU (seccion
   "Retomar") o desde un computador con `python -m tools.evaluation.fase_groq`.
5. Scorecard final y archivos para comitear en `results/`.

**Sesgos del juez y como se controlan.** Familia: el juez es solo Groq; el juez
local Qwen se quito (era el mismo modelo que el baseline y lo preferia en 50-60 de
60 veredictos). Posicion: en el cara a cara cada par se juzga en los dos ordenes y
gana un modelo solo si gana en ambos; la rubrica y el criterio califican una
respuesta a la vez y no tienen posicion. Longitud: los prompts piden no premiar la
extension, se reporta la correlacion puntaje-largo y el juez sin concision.
Reproducibilidad: temperature=0 y seed fija; todo se guarda con checkpoint.

Toda la logica vive en `tools/evaluation/` (con tests); este notebook solo
orquesta. Antes de correr: `Entorno de ejecucion > Cambiar tipo de entorno de
ejecucion > GPU`.

In [ ]:
# Rama del repo de la que se toma el codigo y el dataset. M2 evalua el modelo de
# M1, asi que tiene que correr con EL MISMO dataset con el que se entreno: si se
# clona otra rama, la evaluacion mide contra datos distintos y nada falla a la
# vista.
RAMA = "m3.5"

import os

if not os.path.isdir("/content/Amparo"):
    !git clone -q -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd /content/Amparo && git fetch -q origin && git checkout -q {RAMA} && git pull -q
%cd /content/Amparo

!git log -1 --format="Codigo del commit %h (%ad)" --date=short


In [ ]:
# Dependencias puras de tools/evaluation (metricas clasicas, tests)
!pip install -q -r requirements.txt


In [ ]:
# Stack de ML pesado -- igual que en M1 (m1_finetune.ipynb, celda 1):
# se instala aqui y NO en requirements.txt del repo, para no arriesgar
# reemplazar el build de PyTorch con CUDA que Colab ya trae preinstalado.
# bert-score va aqui tambien (no en requirements.txt) porque depende de
# torch de forma transitiva.
!pip install -q -U transformers peft bitsandbytes accelerate bert-score
!pip install -U "bitsandbytes>=0.46.1"


## Configuracion

Las constantes (seed, val_fraction, modelo base, rutas de Drive) viven en
`tools/evaluation/config.py` -- deben coincidir con `RANDOM_SEED=42` y
`VAL_FRACTION=0.15` de M1 para evaluar sobre el mismo split.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

from tools.evaluation import config, dataset

records = dataset.load_records()
train_records, val_records = dataset.stratified_split(records)
system_prompt = dataset.system_prompt(records)

print(f"Total: {len(records)} | train: {len(train_records)} | val: {len(val_records)}")

# El split de M2 tiene que ser el MISMO que uso M1, o la comparacion baseline vs
# afinado deja de ser valida. Se verifica contra el tamano esperado del dataset
# actual (1536 ejemplos -> 1305/231 con RANDOM_SEED=42 y VAL_FRACTION=0.15).
assert len(records) == 1536, (
    f"El dataset tiene {len(records)} ejemplos y se esperaban 1536: "
    "probablemente se clono una rama con el dataset anterior."
)
assert len(val_records) == 231, "El split no coincide con el de M1 -- revisar RANDOM_SEED/VAL_FRACTION"
print("Split verificado: coincide con el usado en M1.")


## Eval set propio (gold + adversariales)

`data/eval_set.json`: cada caso trae un `criterio` que dice que debe y que no debe
hacer la respuesta; se califica contra ese criterio en la Fase 4 (Groq).

La celda **no deja seguir** si el eval set mide memoria en vez de generalizacion:
- una pregunta identica a una de train (fuga), o
- una casi copia de train (`eval_set.PENDIENTES`), o una pregunta muy parecida a
  una de train (coseno TF-IDF >= 0.55) que nadie reviso. Cada parecida se lee y va
  a `PENDIENTES` (se quita del eval set) o a `REVISADAS_DISTINTAS` (pregunta otra
  cosa, con el motivo).

Tambien informa cuantas repiten preguntas de validacion (no es fuga, pero esos
casos no son propios) y cuantos adversariales hay de cada tipo de abstencion.

In [ ]:
from collections import Counter

from tools.evaluation import eval_set as eval_set_module

eval_records = eval_set_module.load_eval_set()
n_gold = len(eval_set_module.gold_examples(eval_records))
n_adv = len(eval_set_module.adversarial_examples(eval_records))
print(f"Eval set: {len(eval_records)} ejemplos ({n_gold} gold, {n_adv} adversariales)")

solape = eval_set_module.solapamiento(eval_records, train_records, val_records)
assert not solape["en_train"], f"Fuga: preguntas del eval set en train: {solape['en_train']}"
fuga = eval_set_module.fuga_por_parecido(eval_records, train_records)
assert not fuga["pendientes"], (
    f"Casi copias de train que siguen en el eval set (eval_set.PENDIENTES): {fuga['pendientes']}")
assert not fuga["sin_revisar"], "Parecidas a train sin revisar:\n" + "\n".join(
    f"  {p['id']} ~ train {p['train_id']} ({p['similitud']}): {p['pregunta']} / {p['pregunta_train']}"
    for p in fuga["sin_revisar"])
print(f"Sin fuga a train (ni exacta ni casi copia). Repiten preguntas de validacion: "
      f"{len(solape['en_val'])} -> {solape['en_val']}")

tipos = Counter(r["category"] for r in eval_set_module.adversarial_examples(eval_records))
for t in eval_set_module.CATEGORIAS_ADVERSARIALES:
    print(f"  {tipos.get(t, 0):>3}  {t}")

## Fase 1 -- Generacion: baseline (modelo sin fine-tuning)

Con `MAX_NEW_TOKENS_GENERATION = 900` (antes 300, que cortaba el 75 % de las
respuestas del baseline). Se imprime cuantas quedaron cortadas.

In [ ]:
from tools.evaluation import generation

model, tokenizer = generation.load_base_model()

baseline_results = generation.generate_batch(
    model, tokenizer, system_prompt, val_records, label="baseline"
)
print(f"Generadas {len(baseline_results)} respuestas baseline; cortadas por max_new_tokens: {sum(r.cortada for r in baseline_results)}.")


In [ ]:
eval_set_baseline_results = generation.generate_batch(
    model, tokenizer, system_prompt, eval_records, label="baseline"
)
print(f"Eval set (baseline): {len(eval_set_baseline_results)} respuestas; cortadas: {sum(r.cortada for r in eval_set_baseline_results)}.")


## Fase 1b -- Generacion: fine-tuned (adaptador LoRA desde Drive)


In [ ]:
model = generation.attach_adapter(model, config.DRIVE_ADAPTER_DIR)
model.eval()

finetuned_results = generation.generate_batch(
    model, tokenizer, system_prompt, val_records, label="fine_tuned"
)
print(f"Generadas {len(finetuned_results)} respuestas fine-tuned; cortadas: {sum(r.cortada for r in finetuned_results)}.")


In [ ]:
eval_set_finetuned_results = generation.generate_batch(
    model, tokenizer, system_prompt, eval_records, label="fine_tuned"
)
print(f"Eval set (fine-tuned): {len(eval_set_finetuned_results)} respuestas; cortadas: {sum(r.cortada for r in eval_set_finetuned_results)}.")


## Fase 2 -- Metricas sin juez y guardia de rutas y entidades

Se libera la GPU (el juez ya no es local) y se calculan, por respuesta:

- metricas clasicas contra la referencia (F1, BLEU, ROUGE-L, BERTScore, similitud);
- citas numeradas (`domain_metric`);
- **entidades inventadas** (`entity_metric`): nombre con forma de institucion que no
  existe ("Superintendencia de Pensiones", "comisario de policia");
- **rutas incorrectas** (`rutas.py`, reglas): entidad real para un tramite que no le
  toca ("querella ante la Procuraduria", "demanda ante el comisario", juez de
  control de garantias fuera de lo penal, sentencia extranjera sin la Corte
  Suprema...). Las reglas no marcan ninguna de las 1536 referencias del dataset;
- **conceptos fuera de contexto** (`rutas.py`, aprendido del train): entidad o
  figura que las referencias de train nunca usan para el tema de la pregunta.

La guardia se corre tambien sobre las **referencias de validacion**, que son
correctas: lo que marque ahi es ruido, y es la linea base para leer los
porcentajes de cada modelo. Son marcas para revisar, no errores confirmados: la
correccion juridica la juzga Groq (Fase 4), que ademas lista los errores.

In [ ]:
import gc

import torch

from tools.evaluation import pipeline, rutas

del model
gc.collect()
torch.cuda.empty_cache()

mapa = rutas.construir_mapa(train_records)
eval_rows_baseline = pipeline.build_eval_rows(baseline_results, mapa=mapa)
eval_rows_finetuned = pipeline.build_eval_rows(finetuned_results, mapa=mapa)
pipeline.fill_bertscore(eval_rows_baseline)
pipeline.fill_bertscore(eval_rows_finetuned)
all_rows = eval_rows_baseline + eval_rows_finetuned
print(f"Total de filas evaluadas: {len(all_rows)}")

In [ ]:
from tools.evaluation import entity_metric

for etiqueta, filas in (("referencias de validacion (calibracion)", rutas.referencias_como_filas(val_records)),
                        ("baseline", baseline_results), ("fine_tuned", finetuned_results)):
    rep = rutas.reporte(etiqueta, filas, mapa)
    textos = [f["generated"] if isinstance(f, dict) else f.generated for f in filas]
    n_ent = sum(1 for t in textos if entity_metric.find_fabricated_entities(t))
    print(f"\n== {etiqueta}: entidad inventada {round(100 * n_ent / rep.n, 1)} % | "
          f"ruta incorrecta {rep.pct_ruta_incorrecta} % | fuera de contexto {rep.pct_fuera_de_contexto} %")
    print("   reglas:", rep.por_regla or "-")
    print("   fuera de contexto:", rep.por_concepto or "-")

print("\n=== Respuestas del fine-tuned marcadas (revisar a mano) ===")
for r in eval_rows_finetuned:
    if r.marca_entidad or r.marca_ruta or r.marca_contexto:
        print(f"\n[{r.id}] {r.category} | entidad: {r.entidades_inventadas or '-'} | "
              f"ruta: {r.rutas_incorrectas or '-'} | contexto: {r.fuera_de_contexto or '-'}")
        print("   ", r.generated)

## Fase 3 -- Guardar en Drive

Cada corrida va a su propia carpeta con timestamp: respuestas (con el flag de
cortada), metricas por registro y manifiesto. Desde aqui en adelante no hace falta
la GPU: la Fase 4 y el scorecard leen y escriben solo en esta carpeta.

In [ ]:
import json as _json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

from tools.evaluation import scorecard

manifest = pipeline.build_manifest(n_val=len(val_records), hardware=subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip())

run_id = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S")
drive_run_dir = Path(f"{config.DRIVE_EVAL_OUTPUT_ROOT}/{run_id}")
drive_run_dir.mkdir(parents=True, exist_ok=True)


def _guardar_jsonl(nombre, filas):
    (drive_run_dir / nombre).write_text(
        "\n".join(_json.dumps(r.__dict__, ensure_ascii=False) for r in filas) + "\n", encoding="utf-8")


(drive_run_dir / "run_manifest.json").write_text(
    _json.dumps(manifest.__dict__, indent=2, ensure_ascii=False), encoding="utf-8")
_guardar_jsonl("resultados_baseline.jsonl", baseline_results)
_guardar_jsonl("resultados_finetuned.jsonl", finetuned_results)
_guardar_jsonl("eval_set_baseline_results.jsonl", eval_set_baseline_results)
_guardar_jsonl("eval_set_finetuned_results.jsonl", eval_set_finetuned_results)
scorecard.export_csv(drive_run_dir / "metricas_por_registro.csv", all_rows)
print(f"Corrida guardada en: {drive_run_dir}")
print(f"Para retomar otro dia: RETOMAR = \"{run_id}\"")

## Retomar otro dia (sin GPU)

La Fase 4 no cabe en el cupo gratuito de Groq de un dia (toma ~5-7 dias con una
sola key; el limite son los 200 000 tokens diarios). Para seguir otro dia, en
una sesion **sin GPU**: correr las dos primeras celdas (clonar e instalar
`requirements.txt`), montar Drive (`from google.colab import drive;
drive.mount('/content/drive')`), poner aqui el id de la corrida y saltar a la
Fase 4. Si esta es la misma sesion de la generacion, dejar `RETOMAR = ""`.

Tambien se puede correr desde un computador, sin Colab, bajando la carpeta de la
corrida de Drive: `python -m tools.evaluation.fase_groq <carpeta>` (con
`GROQ_API_KEY` en el `.env` local).

In [ ]:
RETOMAR = ""   # ej. "2026-10-07_153000"

if RETOMAR:
    from pathlib import Path
    from tools.evaluation import config
    drive_run_dir = Path(f"{config.DRIVE_EVAL_OUTPUT_ROOT}/{RETOMAR}")
    assert (drive_run_dir / "metricas_por_registro.csv").exists(), f"No existe la corrida {drive_run_dir}"
print(f"Corrida: {drive_run_dir}")

## Fase 4 -- Juez Groq: eval set contra su criterio, rubrica 1-5 y cara a cara

Groq (`openai/gpt-oss-120b`) es de otra familia que Qwen y no escribio ninguna
respuesta. En orden (lo mas valioso por llamada primero):

1. **Eval set contra su criterio** (2 llamadas por caso): cumple / parcial / no cumple y la
   lista de errores juridicos concretos.
2. **Rubrica 1-5 de la validacion** (~462): correccion juridica, prudencia,
   claridad, concision y la **lista de errores juridicos** de cada respuesta. Una
   entidad real que no tramita ese caso cuenta como error grave. El fine-tuned va
   primero.
3. **Cara a cara** (~462): los 231 pares en los dos ordenes; gana un modelo solo si
   gana en ambos.

Requiere `GROQ_API_KEY` en Colab Secrets. Si se acaba el cupo, la celda lo dice y
para; todo lo calificado queda en `<corrida>/checkpoints/` (solo se reusa si el
texto y el prompt son los mismos). Volver a correr la celda al dia siguiente sigue
donde quedo.

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from tools.evaluation import external_judge, fase_groq

external_judge._client = None   # si se cambio la key, el cliente se recrea con la nueva
estado_groq = fase_groq.calificar(drive_run_dir)

In [ ]:
# Eval set contra su criterio, caso por caso (cuando la etapa este completa).
import json as _json

ruta = drive_run_dir / "eval_set_resultados.jsonl"
if ruta.exists():
    for linea in ruta.read_text(encoding="utf-8").splitlines():
        d = _json.loads(linea)
        print(f"\n--- id {d['id']} ({d['tipo']}) {d['category']} ---")
        print(f"Pregunta: {d['query']}")
        print(f"Criterio: {d['criterio']}")
        print(f"[baseline]   {d['baseline']}")
        print(f"  -> {d['baseline_veredicto']} | errores: {d['baseline_errores']}")
        print(f"[fine-tuned] {d['fine_tuned']}")
        print(f"  -> {d['fine_tuned_veredicto']} | errores: {d['fine_tuned_errores']}")
else:
    print("Eval set todavia sin calificar.")

## Fase 5 -- Scorecard final y archivos para el repo

Escribe en la carpeta de la corrida `scorecard.md` y `metricas_por_registro.csv`
con lo que haya. Si a Groq le falta algo, el scorecard lo dice arriba como
**PENDIENTE** (nunca lo muestra como cero).

**Para comitear:** cuando no quede nada pendiente, descargar de la carpeta de la
corrida `scorecard.md`, `metricas_por_registro.csv`, `run_manifest.json`,
`eval_set_resultados.jsonl`, `groq_eval_set_resumen.json`, `groq_validacion.jsonl`
y `groq_cara_a_cara.json` a `results/m2_<fecha>/` en el repo. Los resultados no
pueden vivir solo en Drive (revision docente de M2).

In [ ]:
from tools.evaluation import fase_groq

print(fase_groq.informe(drive_run_dir))
print(f"\nArchivos para results/m2_<fecha>/ en: {drive_run_dir}")

## (Opcional) Explorar resultados en pandas

In [ ]:
import pandas as pd

from tools.evaluation import scorecard

filas = scorecard.load_csv(drive_run_dir / "metricas_por_registro.csv")
df = pd.DataFrame([{**r.__dict__, "judge_sin_concision": r.judge_sin_concision, "n_palabras": r.n_palabras,
                    "marca_ruta": r.marca_ruta, "marca_contexto": r.marca_contexto,
                    "marca_entidad": r.marca_entidad, "con_error_juridico": r.con_error_juridico}
                   for r in filas])
df.groupby("label")[["judge_correccion", "judge_prudencia", "judge_claridad", "judge_concision",
                     "judge_sin_concision", "con_error_juridico", "marca_entidad", "marca_ruta",
                     "marca_contexto", "n_palabras", "cortada", "citation_count"]].mean()